# Scan serialized model artifacts before loading them

This lab creates two synthetic pickle files. The suspicious object's reducer would call a command if someone unpickled it; we never do that. Pickling the object only serializes the recipe, allowing a static scanner to find it safely.

In [ ]:
from hashlib import sha256
from pathlib import Path
import importlib.util
import json
import os
import pickle
import shlex
import subprocess

artifacts = Path("_evidence/model_artifacts")
artifacts.mkdir(parents=True, exist_ok=True)

In [ ]:
class SuspiciousObject:
    def __reduce__(self):
        return (os.system, ("echo MODEL_ARTIFACT_SHOULD_NEVER_EXECUTE",))

safe_path = artifacts / "safe_metadata.pkl"
suspicious_path = artifacts / "suspicious_model.pkl"

with safe_path.open("wb") as f:
    pickle.dump({"model_type": "demo", "weights": [0.1, 0.2]}, f)
with suspicious_path.open("wb") as f:
    pickle.dump(SuspiciousObject(), f)

print("Created files. They will NOT be unpickled in this notebook.")

In [ ]:
def digest(path: Path) -> str:
    return sha256(path.read_bytes()).hexdigest()

inventory = [
    {"path": str(p), "size_bytes": p.stat().st_size, "sha256": digest(p)}
    for p in [safe_path, suspicious_path]
]
inventory

## Run ModelScan in a subprocess

The scanner receives only the quarantined path. Do not import the artifact's Python package or instantiate its model class first.

In [ ]:
modelscan_installed = importlib.util.find_spec("modelscan") is not None
scan_results = []
for path in [safe_path, suspicious_path]:
    cmd = ["modelscan", "-p", str(path)]
    print("\n$", shlex.join(cmd))
    if modelscan_installed:
        result = subprocess.run(cmd, text=True, capture_output=True, check=False)
        output = (result.stdout + "\n" + result.stderr).strip()
        print(output[:12000])
        scan_results.append({"path": str(path), "returncode": result.returncode, "output": output})
    else:
        print("ModelScan is not installed. Use Python 3.12 and: pip install -r requirements.txt")

## Optional: use ModelScan through its Python API

The CLI is easiest to gate in CI. The programmatic API is useful when an artifact admission service needs structured issue counts and severities.

In [ ]:
programmatic_results = []
if modelscan_installed:
    from modelscan.modelscan import ModelScan
    from modelscan.settings import DEFAULT_SETTINGS

    for path in [safe_path, suspicious_path]:
        scanner = ModelScan(settings=DEFAULT_SETTINGS)
        scanner.scan(str(path))
        grouped = scanner.issues.group_by_severity()
        programmatic_results.append({
            "path": str(path),
            "issue_count": len(scanner.issues.all_issues),
            "issues_by_severity": {str(k): len(v) for k, v in grouped.items()},
        })
    programmatic_results
else:
    print("Skipped programmatic API: ModelScan is not installed.")

## Admission policy

Treat scanner errors, unsupported formats, and high-severity findings as blocks. A clean serialization scan is one input; provenance, dependency vulnerabilities, custom code, resource limits, license, and data lineage still require review.

In [ ]:
def admission_decision(scan: dict | None, provenance_verified: bool, format_supported: bool) -> dict:
    if not provenance_verified:
        return {"outcome": "block", "reason": "PROVENANCE_UNVERIFIED"}
    if not format_supported:
        return {"outcome": "block", "reason": "FORMAT_UNSUPPORTED"}
    if scan is None:
        return {"outcome": "block", "reason": "SCAN_NOT_RUN"}

    # ModelScan documents stable CLI exit codes: 0 clear, 1 vulnerabilities,
    # 2 scan error, 3 no supported files, 4 usage error.
    returncode = scan["returncode"]
    if returncode == 0:
        return {"outcome": "allow_to_staging", "reason": "STATIC_SCAN_CLEAR"}
    if returncode == 1:
        return {"outcome": "block", "reason": "VULNERABILITY_FOUND"}
    if returncode == 3:
        return {"outcome": "block", "reason": "NO_SUPPORTED_ARTIFACT"}
    return {"outcome": "block", "reason": "SCAN_ERROR"}

example_without_scanner = admission_decision(None, provenance_verified=True, format_supported=True)
assert example_without_scanner["outcome"] == "block"
print(example_without_scanner)

In [ ]:
evidence = {
    "warning": "Files were created and scanned only; never deserialized.",
    "inventory": inventory,
    "scan_results": scan_results,
    "programmatic_results": programmatic_results,
    "default_when_scanner_missing": example_without_scanner,
}
out = Path("_evidence/07_modelscan_evidence.json")
out.write_text(json.dumps(evidence, indent=2), encoding="utf-8")
print("Wrote", out.resolve())

### Never do this

Do not add `pickle.load`, `torch.load`, `joblib.load`, or an equivalent loader to “confirm whether the warning is real.” Escalate the artifact into an isolated analysis environment or obtain a trusted, verifiable replacement.